<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/Joint%20Angle%20Trial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🏋🏽 Wearable Biomechanics - Joint Angle Trial**
> MARC - Multidisciplinary AI Research Centre | UOP


## **🛠️ Setup OpenDatasetStore**

In [ ]:
%%capture
# @title Store Initialization
from google.colab import drive
import pandas as pd
import os

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Download the latest store manager
!pip install git+https://github.com/janithcyapa/OpenDatasetStore.git

from open_dataset_store import OpenDatasetStore

dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics"
store = OpenDatasetStore(
    base_dir=dataset_path,
    backend="local",
    entity_id_format="sub_{num:04d}",
    entry_id_format="{type}_{num:04d}",
    raw_filename_format="{entry_id}_{entity_id}_{ts}.csv",
    processed_filename_format="{entry_id}_{tag}.parquet"
)


In [9]:
store.summary()
store.list_entries(entry_type="experiments")

📊 Dataset Store Summary
Base Directory : /content/drive/MyDrive/Datasets/Wearable-Biomechanics
Backend        : local
------------------------------
Entities (Total: 13)
  - subjects: 13
------------------------------
Entries (Total: 39)
  - device: 1
  - experiments: 38


,entry_id,entity_id,timestamp,description,raw_csv_path,processed_files,date,time,weight_kg,notes,related_files,processed_metadata
0,experiments_0001,sub_0001,1778229520,Clean & Jerk,raw_data/experiments/experiments_0001_sub_0001...,{'selected_for_4ang': 'processed_data/experime...,2026-05-08,14:08,20,"5 reps, Sensor detached - left ankle last 3 reps",{'front_video': 'related_files/experiments/exp...,{'selected_for_4ang': {'comment': 'No Issues'}}
1,experiments_0002,sub_0001,1778230197,Clean & Jerk,raw_data/experiments/experiments_0002_sub_0001...,{'selected_for_4ang': 'processed_data/experime...,2026-05-08,14:20,20,"5 reps, Sensor detached - left ankle in rep 3",{'raw_zip': 'related_files/experiments/experim...,{'selected_for_4ang': {'comment': 'No Issues'}}
2,experiments_0003,sub_0002,1778666861,Clean & Jerk,raw_data/experiments/experiments_0003_sub_0002...,{},2026-05-13,15:37,20,"5 reps, Device 6 (Right Arm and wrist) are mis...",{'raw_zip': 'related_files/experiments/experim...,{}
3,experiments_0004,sub_0002,1778667119,Clean & Jerk,raw_data/experiments/experiments_0004_sub_0002...,{},2026-05-13,15:42,20,"5 reps, Device 6 (Right Arm and wrist) are mis...",{'raw_zip': 'related_files/experiments/experim...,{}
4,experiments_0005,sub_0003,1778670186,Clean & Jerk,raw_data/experiments/experiments_0005_sub_0003...,{},2026-05-13,16:33,20,"5 reps, Device 6 (Right Arm and wrist) are mis...",{'raw_zip': 'related_files/experiments/experim...,{}
5,experiments_0006,sub_0003,1778671087,Clean & Jerk,raw_data/experiments/experiments_0006_sub_0003...,{},2026-05-13,16:48,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...,{}
6,experiments_0007,sub_0004,1781068783,Clean & Jerk,raw_data/experiments/experiments_0007_sub_0004...,{'selected_for_4ang': 'processed_data/experime...,2026-06-10,10:49,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...,{'selected_for_4ang': {'comment': 'Error in 4t...
7,experiments_0008,sub_0004,1781069072,Clean & Jerk,raw_data/experiments/experiments_0008_sub_0004...,{'selected_for_4ang': 'processed_data/experime...,2026-06-10,10:54,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...,{'selected_for_4ang': {'comment': 'No Issues'}}
8,experiments_0009,sub_0004,1781069235,Clean & Jerk,raw_data/experiments/experiments_0009_sub_0004...,{'selected_for_4ang': 'processed_data/experime...,2026-06-10,10:57,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...,{'selected_for_4ang': {'comment': 'Error in IM...
9,experiments_0010,sub_0004,1781069666,Clean & Jerk,raw_data/experiments/experiments_0010_sub_0004...,{'selected_for_4ang': 'processed_data/experime...,2026-06-10,11:04,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...,{'selected_for_4ang': {'comment': 'Error in 3r...


# Visual Estimation

In [10]:
entry_type = "experiments"
entry_id = "experiments_0001"
label = "front_video"
label2 = "side_video"

local_path = store.download_related_file(
    entry_type=entry_type,
    entry_id=entry_id,
    label=label,
    dest_folder="."
)


📥 Downloaded to: ./experiments_0001_sub_0001_front_video.mp4
File downloaded to: ./experiments_0001_sub_0001_front_video.mp4


In [13]:
!pip install -q ultralytics opencv-python pandas scipy
import ultralytics
print("Ultralytics version:", ultralytics.__version__)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics version: 8.4.163


In [14]:
# @title
import json
import cv2
from ultralytics import YOLO

# Load the model (downloads automatically on first execution)
model = YOLO('yolo11x-pose.pt')

def extract_keypoints(video_path, output_json_path):
    cap = cv2.VideoCapture(video_path)
    all_frames_data = []
    frame_idx = 0

    print(f"Processing: {video_path}")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # verbose=False suppresses per-frame console logging
        results = model(frame, verbose=False)

        frame_data = {"frame": frame_idx, "keypoints": [], "scores": []}

        if len(results) > 0 and results[0].keypoints is not None:
            keypoints = results[0].keypoints.xy[0].cpu().numpy().tolist()
            scores = results[0].keypoints.conf[0].cpu().numpy().tolist()

            frame_data["keypoints"] = keypoints
            frame_data["scores"] = scores

        all_frames_data.append(frame_data)
        frame_idx += 1

    cap.release()

    with open(output_json_path, "w") as f:
        json.dump(all_frames_data, f)
    print(f"Saved {frame_idx} frames to {output_json_path}")

# Run extraction on both un-synced feeds
extract_keypoints("/content/experiments_0001_sub_0001_front_video.mp4", "/content/front_keypoints.json")


Processing: /content/experiments_0001_sub_0001_front_video.mp4
Saved 2375 frames to /content/front_keypoints.json


In [15]:
# @title
import json
import cv2
import numpy as np

video_path = "/content/experiments_0001_sub_0001_front_video.mp4"
json_path = "/content/front_keypoints.json"
output_video_path = "/content/front_visualized.mp4"

# Standard COCO 17-keypoint skeleton connections (pairs of joint indices)
SKELETON_PAIRS = [
    (0, 1), (0, 2), (1, 3), (2, 4),             # Facial connections
    (5, 6),                                     # Clavicle / Shoulders
    (5, 7), (7, 9),                             # Left arm
    (6, 8), (8, 10),                            # Right arm
    (5, 11), (6, 12),                           # Torso lateral bounds
    (11, 12),                                   # Pelvic base
    (11, 13), (13, 15),                         # Left leg
    (12, 14), (14, 16)                          # Right leg
]

# Read coordinate dataset
with open(json_path, "r") as f:
    keypoints_data = json.load(f)

cap = cv2.VideoCapture(video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Initialize MP4 writer using modern H.264/MP4V codec
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

frame_idx = 0
CONFIDENCE_THRESHOLD = 0.4  # Minimum threshold to display point

print("Rendering visual skeleton overlay...")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret or frame_idx >= len(keypoints_data):
        break

    frame_info = keypoints_data[frame_idx]
    kpts = frame_info.get("keypoints", [])
    scores = frame_info.get("scores", [])

    if len(kpts) == 17:
        # Draw skeleton limbs
        for p1, p2 in SKELETON_PAIRS:
            if scores[p1] >= CONFIDENCE_THRESHOLD and scores[p2] >= CONFIDENCE_THRESHOLD:
                pt1 = (int(kpts[p1][0]), int(kpts[p1][1]))
                pt2 = (int(kpts[p2][0]), int(kpts[p2][1]))
                cv2.line(frame, pt1, pt2, (0, 255, 0), 2, lineType=cv2.LINE_AA)

        # Draw anatomical keypoints
        for i, (x, y) in enumerate(kpts):
            if scores[i] >= CONFIDENCE_THRESHOLD:
                cv2.circle(frame, (int(x), int(y)), 4, (0, 0, 255), -1, lineType=cv2.LINE_AA)

    # Frame counter watermark
    cv2.putText(frame, f"Frame: {frame_idx}", (25, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, lineType=cv2.LINE_AA)

    out.write(frame)
    frame_idx += 1

cap.release()
out.release()
print(f"Visualization complete: {output_video_path}")

Rendering visual skeleton overlay...
Visualization complete: /content/front_visualized.mp4


In [16]:
from IPython.display import HTML
from base64 import b64encode

# Convert video to H.264 via ffmpeg to guarantee web browser decoding compatibility
!ffmpeg -y -loglevel panic -i /content/front_visualized.mp4 -vcodec libx264 -crf 28 /content/front_browser.mp4

mp4 = open('/content/front_browser.mp4', 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=640 controls>
      <source src="{data_url}" type="video/mp4">
</video>
""")